# 00 · Coorte

Monta a coorte do estudo e confere os caminhos das imagens.

**Critérios**
- **BRSET**: só pacientes com diabetes (texto de `comorbidities`; a coluna `diabetes` codifica dado ausente como "No"); imagens com grau ICDR; foco e iluminação adequados.
- **mBRSET**: só fotos centradas na mácula (arquivos `.1` = olho direito e `.3` = olho esquerdo); imagens com grau ICDR; `final_quality == "yes"`.
- **Ambas**: todas as imagens elegíveis (os dois olhos; ICs e testes agrupados por paciente); 50 pacientes por base vão para o piloto (só ajuste de prompt e parser).
- **Desfecho primário**: grau ICDR (0–4) previsto pelo modelo vs. o de referência (kappa ponderado quadrático).
- **Desfecho secundário**: RD referenciável = ICDR ≥ 2 e/ou edema macular.

Caminhos em `config.py`. Nada de linhas individuais é exibido: os dados são de acesso controlado.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import config
pd.set_option("display.max_columns", 50, "display.width", 250, "display.max_rows", 200)
from vlmdr import cohort

## Arquivos de entrada

In [ ]:
for name in ["BRSET_LABELS", "MBRSET_LABELS", "BRSET_IMAGES", "MBRSET_IMAGES", "EMB_NPY_DIR"]:
    p = getattr(config, name)
    print(f"{name:14s} {'ok ' if p.exists() else 'NÃO ENCONTRADO'}  {p}")

## Regra de diabetes no BRSET

Compara o texto de `comorbidities` com a coluna `diabetes`, por paciente. Confira antes de seguir.

In [ ]:
b = cohort.read_brset()
by_text = b.comorbidities.fillna("").str.lower().str.contains("diab").groupby(b.patient_id).max()
col = b.diabetes.astype(str).str.strip().str.lower().groupby(b.patient_id).first()
pd.crosstab(col.rename("coluna diabetes"), by_text.rename("comorbidities contém 'diab'"), margins=True)

## Fluxo de exclusão e resumo

In [ ]:
parts, flow, summ = cohort.build()
display(flow, summ)

## Distribuição do desfecho e dos graus (amostra de avaliação)

In [ ]:
for ds, d in parts.items():
    ev = d[d.split == "eval"]
    print(f"\n{ds}: {len(ev)} imagens de {ev.patient_id.nunique()} pacientes, {ev.referable.sum()} referenciáveis "
          f"({100 * ev.referable.mean():.1f}%)")
    display(pd.crosstab(ev.grade, ev.macular_edema.rename("edema"), margins=True))

## Imagens no disco

In [ ]:
for ds, d in parts.items():
    ok = d.image_path.map(lambda p: Path(p).exists())
    print(f"{ds}: {ok.sum()}/{len(d)} imagens encontradas")
    if not ok.all():
        print("  exemplo de caminho ausente:", d.loc[~ok, "image_path"].iloc[0])